# ReAct Agent

In [3]:
import openai
import re
import httpx
import os 

# using Ollama instead of openAI
import ollama
from ollama import Client
llm = "qwen3"

In [4]:
client = Client()

In [13]:
chat_completion = client.chat(
    model="qwen3",
    messages=[{"role":"user", "content": "Hello world!"}]
)

In [14]:
chat_completion.message.content

"Hello! 😊 How can I assist you today? Whether you have questions, need help with something, or just want to chat, I'm here for you!"

In [73]:
class Agent: 
    def __init__(self, system=""):
        self.system=system
        self.messages = []
        if self.system:
            self.messages.append({"role":"system", "content":system})
    def __call__(self, message):
        self.messages.append({"role":"user", "content":message})
        result = self.execute()
        self.messages.append({"role":"assistant", "content":result})
        return result 
    def execute(self):
        completion = client.chat(
              model="qwen3",
              messages=self.messages,
            options= {
                "temperature": 0,
               
               
            }
        )
        return completion.message.content


In [61]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [82]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    name = name.strip()
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}
    

In [25]:
abot = Agent(prompt)

In [26]:
result = abot("How much does a toy poodle weight?")
print(result)

Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight function.
Action: average_dog_weight: Toy Poodle
PAUSE.


In [27]:
result = average_dog_weight("Toy Poodle")
result

'a toy poodles average weight is 7 lbs'

In [30]:
next_prompt = f"Observation:{result}"
abot(next_prompt)

'Answer: A toy poodle weighs approximately 7 lbs on average.'

In [31]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session. Adhere strictly to this structure:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE. \n\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role

In [62]:
abot = Agent(prompt)

question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
print(abot(question))

Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together.  
Action: average_dog_weight: Border Collie  



In [63]:
result = average_dog_weight("Border Collie")
next_prompt = f"Observation:{result}"
abot(next_prompt)


"Thought: Now I need the average weight of a Scottish Terrier to add to the Border Collie's weight.  \nAction: average_dog_weight: Scottish Terrier  \n"

In [64]:
result = average_dog_weight("Scottish Terrier")
next_prompt = f"Observation:{result}"
abot(next_prompt)

'Action: calculate: 37 + 20  \n'

##### loop
action_re = re.compile('Action: (\w+): (.*)$')

In [83]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i+=1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions: 
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}")
            print(f"-- running {action}{action_input}")
            observation = known_actions[action](action_input)
            print(f"Observation: {observation}")
            next_prompt = f"Observation: {observation}"
        else:
            return

In [84]:
query(question)

Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together.  
Action: average_dog_weight: Border Collie  
PAUSE  


-- running average_dog_weightBorder Collie  
Observation: a Border Collies average weight is 37 lbs
Thought: Now I need the average weight of a Scottish Terrier to add to the Border Collie's weight.  
Action: average_dog_weight: Scottish Terrier  
PAUSE  


-- running average_dog_weightScottish Terrier  
Observation: Scottish Terriers average 20 lbs
Thought: Now I can calculate the combined weight by adding the two average weights.  
Action: calculate: 37 + 20  
PAUSE  


-- running calculate37 + 20  
Observation: 57
Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.
